<a href="https://colab.research.google.com/github/Kalyani-asekar/NNDL/blob/main/practical8nndl_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(42)
X = torch.randn(1000, 64).to(device)

class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, input_dim)
        )

    def forward(self, x):
        latent = self.encoder(x)
        reconstruction = self.decoder(latent)
        return reconstruction, latent

input_dim = 64
latent_dim = 8
model = Autoencoder(input_dim, latent_dim).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 100
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    reconstructed, _ = model(X)
    loss = criterion(reconstructed, X)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.6f}")

model.eval()
with torch.no_grad():
    reconstructed, latent_features = model(X[:1])
    print("\nOriginal Input Shape:", X[:1].shape)
    print("Latent Features Shape (Reduced Dimension):", latent_features.shape)
    print("Reconstructed Output Shape:", reconstructed.shape)
    print("\nExtracted Latent Features:\n", latent_features.cpu().numpy())